# 01 — Quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-systems-today/cubespec/blob/main/notebooks/01_quickstart.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ai-systems-today/cubespec/main?urlpath=lab/tree/notebooks/01_quickstart.ipynb) [![nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/ai-systems-today/cubespec/blob/main/notebooks/01_quickstart.ipynb) [![Dashboard](https://img.shields.io/badge/open-dashboard-2563eb)](https://sensitive-spark.lovable.app)

**Abstract.** Run one Monte-Carlo experiment end-to-end on the default CSP, then visualise the P9 distribution and its 95% bootstrap CI.

**Mirrors.** **Live MC** tab — the default behaviour you see when you press *Start* without changing any controls.


In [ ]:
# cubespec-bootstrap
# Install cubespec on first run (Colab/Binder safe; no-op if already importable).
try:
    import cubespec  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/ai-systems-today/cubespec.git'])
    import cubespec  # noqa: F401


## 1. Method

We draw `n = 50 000` independent samples of the seven-parameter Cube Specification (CSP), push them through the calibrated Poly2-ridge surrogate `f(X) → (P7, P8, P9)`, and report the mean, standard deviation and 95% bootstrap confidence interval on the mean of `P9` (compressive strength, MPa).

Bit-for-bit reproducible thanks to the mulberry32 PRNG — same seed in Python, in the dashboard, and in the CI parity test.


In [ ]:
from cubespec import (
    DEFAULT_CSP, sample_independent, compute_outputs_batch, bootstrap_mean_ci,
)
import numpy as np

SEED = 1337
N = 50_000
X = sample_independent(DEFAULT_CSP, n=N, seed=SEED)
Y = compute_outputs_batch(X)
P9 = Y[:, 2]
print(f'P9 mean = {P9.mean():.3f} MPa')
print(f'P9 sd   = {P9.std(ddof=1):.3f} MPa')
ci = bootstrap_mean_ci(P9, B=1000, seed=SEED)
print(f'95% CI on P9 mean = [{ci.lo:.3f}, {ci.hi:.3f}] MPa  '
      f'(width {ci.hi - ci.lo:.3f})')


## 2. P9 distribution with the bootstrap CI overlaid

The histogram is the empirical distribution of `P9` across the Monte-Carlo run. The blue line is the sample mean; the red band is the 95% confidence interval **on the mean** (not on individual samples — that is the much wider P5–P95 prediction interval).


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.hist(P9, bins=60, color='#5b8def', alpha=0.8, edgecolor='white')
ax.axvline(P9.mean(), color='#2952b3', lw=2, label='mean')
ax.axvspan(ci.lo, ci.hi, color='#e94f64', alpha=0.25, label='95% CI on mean')
ax.set_xlabel('P9 compressive strength (MPa)')
ax.set_ylabel('count')
ax.set_title(f'P9 distribution, n = {N:,}, seed = {SEED}')
ax.legend()
fig.tight_layout()
plt.show()


## 3. Interpretation

The bell-shaped histogram tells us the surrogate is well-behaved: no heavy tails, no multimodality. The CI on the mean is narrow (width ≪ 1 MPa for n = 50k) which is what a thesis reader expects from a Monte-Carlo estimate at this sample size. If you halve n the CI roughly √2× wider — verify it as a sanity check.

## 4. Try this

Re-run with `N = 5_000` and `N = 500_000` and watch the CI width shrink as 1/√N.


In [ ]:
# cubespec-reproducibility-footer
import platform, sys, numpy as np, scipy, pandas as pd, cubespec
print('Reproducibility')
print('  cubespec :', cubespec.__version__)
print('  python   :', sys.version.split()[0], 'on', platform.system())
print('  numpy    :', np.__version__)
print('  scipy    :', scipy.__version__)
print('  pandas   :', pd.__version__)
